# CPIoT-XAD2025 — end-to-end reference run

Hybrid CNN/LSTM cross-domain anomaly detection on a row-aligned fusion of network-telemetry (`*_nbaiot`) and physical-process (`*_swat`) features.

This notebook runs the same pipeline as the command-line entry point. The protocol — leakage-safe episode-aware splitting, a two-stage calibration holdout, validation-only model selection — is described in [`docs/PROTOCOL.md`](https://github.com/Bature82/cpiot-xad2025-hybrid-cnn-lstm/blob/main/docs/PROTOCOL.md).

**Runtime:** select a GPU runtime (`Runtime → Change runtime type → GPU`). The LSTM stream is slow on CPU.

## 1. Setup

In [ ]:
import os

if not os.path.exists('cpiot-xad2025-hybrid-cnn-lstm'):
    !git clone https://github.com/Bature82/cpiot-xad2025-hybrid-cnn-lstm.git
%cd cpiot-xad2025-hybrid-cnn-lstm
!pip install -q -r requirements.txt

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "NONE — enable a GPU runtime")

## 2. Data

The pipeline expects one CSV where each row is an aligned timestep, with network-stream columns ending in `_nbaiot`, process-stream columns ending in `_swat`, and a binary label column. **Row order is time order.**

The corpus is not redistributed with this repository. If your copy lives in Google Drive, mount it:

In [ ]:
try:
    from google.colab import drive
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except ImportError:
    print('Not running on Colab — skipping Drive mount.')

In [ ]:
DATA_PATH  = '/content/drive/MyDrive/Fusion/fused_output_v5/CPIoT-XAD2025_fused_hybrid_v5.csv'
RESULT_DIR = 'results'

## 3. Smoke run

One mode, one seed, three epochs — enough to confirm the corpus parses, the splits are viable and the outputs land where they should. Run this before committing to the full protocol.

In [ ]:
!python -m cpiot_xad.cli \
    --data-path "$DATA_PATH" \
    --result-dir "$RESULT_DIR/smoke" \
    --modes fused --seeds 42 --epochs 3

## 4. Full protocol

Three ablation modes × five seeds, each with the complete selection grid and all four operating points. This takes hours on a GPU; it is resumable, so re-running the identical command after a disconnect picks up from the first unfinished run.

In [ ]:
!python -m cpiot_xad.cli \
    --data-path "$DATA_PATH" \
    --result-dir "$RESULT_DIR" 

### Or call the API directly

Useful when you want the returned rows in memory rather than on disk.

In [ ]:
from cpiot_xad.config import CFG
from cpiot_xad.data import load_corpus
from cpiot_xad.protocol import run_protocol

CFG['RESULT_DIR'] = RESULT_DIR
CFG['EPOCHS'] = 3          # raise for a real run
os.makedirs(RESULT_DIR, exist_ok=True)

X_np, y_rows, nb_cols, sw_cols = load_corpus(DATA_PATH)
rows, grid, ops, sa, sk, ss = run_protocol(
    'fused', 42, X_np, y_rows, len(nb_cols), len(sw_cols), make_figures=True)

import pandas as pd
pd.DataFrame(rows)[['scorer', 'TEST_Accuracy', 'TEST_Precision',
                    'TEST_Recall', 'TEST_F1', 'TEST_ROC-AUC', 'TEST_PR-AUC']]

## 5. Results

`metrics_summary.csv` carries mean / std / min / max / median across seeds; `best_runs.csv` names the single best run per configuration, so a headline number can be quoted without passing it off as the average.

In [ ]:
import pandas as pd

summary = pd.read_csv(f'{RESULT_DIR}/metrics_summary.csv', header=[0, 1], index_col=[0, 1])
cols = [c for c in [('TEST_Accuracy', 'mean'), ('TEST_Precision', 'mean'),
                    ('TEST_Recall', 'mean'), ('TEST_F1', 'mean'),
                    ('TEST_ROC-AUC', 'mean'), ('TEST_PR-AUC', 'mean')]
        if c in summary.columns]
summary[cols].round(4)

In [ ]:
ops = pd.read_csv(f'{RESULT_DIR}/operating_points.csv')
(ops.groupby(['mode', 'scorer', 'target_FPR'])
    [['TEST_Precision', 'TEST_Recall', 'TEST_F1']].mean().round(4))

### Figures from the primary seed

In [ ]:
from IPython.display import Image, display
import glob

for f in sorted(glob.glob(f'{RESULT_DIR}/*.png')):
    print(f)
    display(Image(f))